# 🧪 ДЗ: Трекинг объектов — KLT и Deep SORT (автопроверка)
**Формат**: заполнить ячейки `YOUR CODE HERE`, выполнить ноутбук сверху вниз.
**Выход**: в конце печатается **JSON одной строкой** с итоговым баллом по 100-балльной шкале.


## 0) Информация о студенте

In [1]:
# === ОБЯЗАТЕЛЬНО ЗАПОЛНИТЬ ===
full_name = "Doe John"     # например: "Тощев Александр"
student_group = "11-111"      # например: "208"
assignment_id = "HW_Tracking_KLT_DeepSORT_Autograde"
assert full_name != "Фамилия Имя", "Заполните full_name"
assert student_group != "Группа", "Заполните student_group"
print("✔ Student Info OK")

print("Student:", full_name)

✔ Student Info OK
Student: Doe John


## 0.1) Дедлайны и время сдачи
- Установите окна сдачи в формате `YYYY-MM-DD HH:MM` (локальное время).
- **Правило штрафа**: после дедлайна баллы линейно уменьшаются и к концу окна
  (END_DATE + (END_DATE - START_DATE)) становятся **0**.
- Если доступно время последнего изменения файла, используем его, иначе — текущее UTC.


In [3]:

# Colab / Jupyter-ready cell
# Home assignment auto-checker: Feature detection & matching
# Usage: student edits cells marked with "=== YOUR CODE HERE ==="
# Instructor: set DUE_DATE (ISO string) and LATE_WINDOW_DAYS

# 1) Настройки (инструктор)
from datetime import datetime, timezone, timedelta
import json
import os
import sys
from pathlib import Path

from datetime import datetime, timezone, timedelta

# Установите окна приёма (пример):

start_at_iso = "2025-10-21T09:00-04:00"  #@param {type:"string"}
due_at_iso   = "2025-11-04T23:59-04:00"  #@param {type:"string"}
start_dt = datetime.fromisoformat(start_at_iso)
due_dt   = datetime.fromisoformat(due_at_iso)
# Для протокола: время сдачи берём текущее (можно заменить на mtime файла)
import os
from datetime import datetime, timezone

# 📅 Add submission date based on file modification time
try:
    nb_path = __file__ if "__file__" in globals() else "Feature_Matching_Autograder.ipynb"
    mtime = os.path.getmtime(nb_path)
    submission_dt = datetime.fromtimestamp(mtime, tz=timezone.utc)
except Exception:
    submission_dt = datetime.utcnow().replace(tzinfo=timezone.utc)



# 2) Установим зависимости (OpenCV, scikit-image и т.д.)
# В Colab: раскомментируйте и выполните
# !pip install opencv-contrib-python scikit-image numpy pytest

import numpy as np
import cv2
from skimage.metrics import structural_similarity as ssim
import tempfile
import math
import time

# 3) Helper: читает время сдачи. В Colab студент указывает строкой (или можно взять mtime файла)
def parse_iso_datetime(s):
    # robust parse of ISO-like strings
    return datetime.fromisoformat(s)



# 4) Задачи (инструктор описывает, студент реализует)

print("Due:", due_dt, "| Submission time detected:", submission_dt.isoformat())

Due: 2025-11-04 23:59:00-04:00 | Submission time detected: 2025-10-06T13:27:28.615453+00:00


## 1) Генерация тестового видео
Создаём `synthetic_demo.mp4` (несколько движущихся объектов).

In [4]:
import cv2, numpy as np, random
w, h, fps, seconds, num_objs = 640, 360, 25, 12, 6
objs = []
for i in range(num_objs):
    x = random.randint(50, w-50)
    y = random.randint(50, h-50)
    vx = random.choice([-3, -2, -1, 1, 2, 3])
    vy = random.choice([-3, -2, -1, 1, 2, 3])
    shape = random.choice(['circle', 'rect'])
    size = random.randint(12, 24)
    objs.append([x, y, vx, vy, shape, size])
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('synthetic_demo.mp4', fourcc, fps, (w, h))
for t in range(fps*seconds):
    frame = np.zeros((h, w, 3), dtype=np.uint8)
    noise = np.random.randint(0, 10, (h, w, 1), dtype=np.uint8)
    frame[:] = noise
    for i in range(num_objs):
        x, y, vx, vy, shape, size = objs[i]
        if x < size or x > w-size: vx *= -1
        if y < size or y > h-size: vy *= -1
        x += vx; y += vy
        objs[i][0], objs[i][1], objs[i][2], objs[i][3] = x, y, vx, vy
        if shape == 'circle':
            cv2.circle(frame, (int(x), int(y)), size, (50+20*i, 180-20*i, 120+10*i), -1)
        else:
            cv2.rectangle(frame, (int(x-size), int(y-size)), (int(x+size), int(y+size)), (150-20*i, 50+20*i, 200-10*i), -1)
    out.write(frame)
out.release()
print('Saved synthetic_demo.mp4')


Saved synthetic_demo.mp4


## 2) KLT: реализуйте трекинг точек (PyrLK)
- Заполните фрагмент `YOUR CODE HERE` так, чтобы:
  1) Создавался ролик `klt_tracks.mp4` с треками;
  2) В список `KLT_POINTS_PER_FRAME` записывалось число активных точек на каждом кадре;
  3) В `KLT_LONGEST_TRACK_TIMELINE` сохранялась история **одной самой длинной** точки (ID/индекс или `None`).


In [6]:
import cv2, numpy as np
video_path = 'synthetic_demo.mp4'
cap = cv2.VideoCapture(video_path)
feature_params = dict(maxCorners=300, qualityLevel=0.2, minDistance=7, blockSize=7)
lk_params = dict(winSize=(21,21), maxLevel=3,
                 criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 20, 0.03))
ret, old_frame = cap.read()
assert ret, 'Не удалось прочитать видео'
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
mask = np.zeros_like(old_frame)
reseed_interval = 10
frame_idx = 0
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('klt_tracks.mp4', fourcc, 25, (old_frame.shape[1], old_frame.shape[0]))

# === Для оценки ===
KLT_POINTS_PER_FRAME = []
KLT_LONGEST_TRACK_TIMELINE = []  # список индексов (int) или None по кадрам
current_ids = None  # массив ID для p0
next_id = 0
tracks_history = {}

def assign_ids(num):
    global next_id
    ids = np.arange(next_id, next_id+num)
    next_id += num
    return ids.reshape(-1,1)

if p0 is not None:
    current_ids = assign_ids(len(p0))

while True:
    ret, frame = cap.read()
    if not ret: break
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # 1) убедимся, что есть точки для трекинга
    if p0 is None or len(p0) == 0:
        p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
        current_ids = assign_ids(len(p0)) if p0 is not None else None
    
    # 2) считаем оптический поток (один раз, без walrus)
    p1, st, err = # YOU CODE HERE 
    # 3) валидируем результат: должны быть и p1, и st, и хотя бы одна «живая» точка
    valid = (p1 is not None) and (st is not None) and np.any(st == 1)
    
    if not valid:
        # пересемплируем точки и делаем служебные обновления
        p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
        current_ids = assign_ids(len(p0)) if p0 is not None else None
        old_gray = frame_gray.copy()
        mask[:] = 0  # быстрее, чем np.zeros_like каждый раз
        frame_idx += 1
        KLT_POINTS_PER_FRAME.append(0)
        KLT_LONGEST_TRACK_TIMELINE.append(None)
        continue
    
    # 4) выбираем валидные соответствия
    good_new = p1[st == 1]
    good_old = p0[st == 1]
    good_ids = current_ids[st == 1] if current_ids is not None else None

    # Рисование и сбор статистики
    for (new, old, gid) in zip(good_new, good_old, good_ids.reshape(-1)):
        a,b = new.ravel().astype(int)
        c,d = old.ravel().astype(int)
        cv2.line(mask, (a,b), (c,d), (0,255,0), 1)
        cv2.circle(frame, (a,b), 2, (0,0,255), -1)
        tracks_history.setdefault(int(gid), []).append(frame_idx)

    img = cv2.add(frame, mask)
    out.write(img)

    KLT_POINTS_PER_FRAME.append(len(good_new))

    # Обновление
    old_gray = frame_gray.copy()
    p0 = good_new.reshape(-1,1,2)
    current_ids = good_ids.reshape(-1,1)

    # re-seed
    frame_idx += 1
    if frame_idx % reseed_interval == 0:
        more = # YOU CODE HERE
        if more is not None:
            extra_ids = assign_ids(len(more))
            p0 = np.concatenate([p0, more], axis=0)
            current_ids = np.concatenate([current_ids, extra_ids], axis=0)

cap.release(); out.release()

# Выберем самую длинную историю по количеству кадров
if tracks_history:
    best_id = max(tracks_history, key=lambda k: len(tracks_history[k]))
else:
    best_id = None

# Построим таймлайн best_id: если на кадре он активен — ставим его id, иначе None
if best_id is not None:
    max_frame = max([max(v) for v in tracks_history.values()]) if tracks_history else frame_idx
    present = set(tracks_history[best_id])
    KLT_LONGEST_TRACK_TIMELINE = [best_id if i in present else None for i in range(max_frame+1)]
else:
    KLT_LONGEST_TRACK_TIMELINE = []

len(KLT_POINTS_PER_FRAME), len(KLT_LONGEST_TRACK_TIMELINE)


(299, 299)

## 3) Deep SORT + YOLOv8 (в Colab)
- Установите пакеты: `ultralytics`, `deep-sort-realtime`, `filterpy`, `lapx`, `tqdm`.
- Запустите функцию `run_deepsort_yolo(...)` ниже.
- Сохраните список `DS_ID_TIMELINE` — ID одного подтверждённого трека (например, самого «старого» на кадре).

In [7]:
# !pip -q install ultralytics deep-sort-realtime filterpy lapx tqdm
import numpy as np
try:
    from ultralytics import YOLO
    from deep_sort_realtime.deepsort_tracker import DeepSort
    import cv2
    HAVE_DS = True
except Exception as e:
    HAVE_DS = False
    print('Deep SORT/YOLO недоступны, пропустим шаг. Причина:', e)

DS_ID_TIMELINE = []  # заполните ID одного трека по кадрам (или None)

if HAVE_DS:
    def run_deepsort_yolo(video_in='synthetic_demo.mp4', out_path='deepsort_yolo.mp4', conf=0.25, classes=None):
        model = YOLO('yolov8n.pt')
        tracker = DeepSort(max_age=30, n_init=3, nms_max_overlap=1.0, max_cosine_distance=0.4)
        cap = cv2.VideoCapture(video_in)
        assert cap.isOpened()
        w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(out_path, fourcc, fps, (w, h))
        oldest_id = None
        while True:
            ret, frame = cap.read()
            if not ret: break
            results = model.predict(frame, conf=conf, verbose=False)[0]
            dets = []
            for b in results.boxes:
                cls_id = int(b.cls.item())
                if (classes is not None) and (cls_id not in classes):
                    continue
                x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
                score = float(b.conf.item())
                dets.append(([x1, y1, x2-x1, y2-y1], score, cls_id))
            tracks = tracker.update_tracks(dets, frame=frame)
            active_ids = []
            for tr in tracks:
                if not tr.is_confirmed() or tr.time_since_update > 0: continue
                active_ids.append(tr.track_id)
            if active_ids:
                oldest_id = active_ids[0] if oldest_id is None else oldest_id
                DS_ID_TIMELINE.append(oldest_id if oldest_id in active_ids else active_ids[0])
            else:
                DS_ID_TIMELINE.append(None)
            out.write(frame)
        cap.release(); out.release()
        return out_path
    # Пример запуска (раскомментируйте):
    # run_deepsort_yolo()
else:
    # Фолбэк: пустой таймлайн, чтобы не ломать автопроверку
    DS_ID_TIMELINE = []
HAVE_DS, len(DS_ID_TIMELINE)


Deep SORT/YOLO недоступны, пропустим шаг. Причина: No module named 'ultralytics'


(False, 0)

## 4) Метрика непрерывности трека
Функция считает среднюю длину непрерывных сегментов. Заполните таймлайны выше.

In [8]:
def continuity_metric(id_timeline):
    segs = []
    cur_id, cur_len = None, 0
    for tid in id_timeline:
        if tid is None:
            if cur_len > 0:
                segs.append(cur_len)
            cur_id, cur_len = None, 0
            continue
        if cur_id is None or tid != cur_id:
            if cur_len > 0:
                segs.append(cur_len)
            cur_id, cur_len = tid, 1
        else:
            cur_len += 1
    if cur_len > 0:
        segs.append(cur_len)
    avg_len = float(sum(segs))/len(segs) if segs else 0.0
    return avg_len, segs

KLT_CONT_AVG, KLT_SEGS = continuity_metric(KLT_LONGEST_TRACK_TIMELINE)
DS_CONT_AVG, DS_SEGS   = continuity_metric(DS_ID_TIMELINE)
KLT_CONT_AVG, len(KLT_SEGS), DS_CONT_AVG, len(DS_SEGS)


(299.0, 1, 0.0, 0)

## 6) Автопроверка и подсчёт баллов
Правила:
- (30) KLT: создан ролик + есть точки на кадрах + построен таймлайн самой длинной точки
- (30) Deep SORT: заполнен `DS_ID_TIMELINE` (если пакетов нет — 0 за раздел)
- (20) Метрики: корректно посчитана `continuity_metric()` и ненулевые сегменты (где применимо)
- (20) Текст: присутствуют ключевые слова (`перекрыт/occlusion`, `Re-ID`, `Kalman/Hungarian`) — эвристическая проверка
Штраф: после дедлайна линейное уменьшение до 0 к концу окна (END + (END-START)).


In [9]:
import re, math, json
raw = 0
details = {}

# --- KLT (30) ---
klt_ok_video = os.path.exists('klt_tracks.mp4')
klt_ok_points = isinstance(KLT_POINTS_PER_FRAME, list) and (sum(KLT_POINTS_PER_FRAME) > 0)
klt_ok_timeline = isinstance(KLT_LONGEST_TRACK_TIMELINE, list) and (len(KLT_LONGEST_TRACK_TIMELINE) > 0)
klt_score = 0
if klt_ok_video: klt_score += 10
if klt_ok_points: klt_score += 10
if klt_ok_timeline and (sum(1 for x in KLT_LONGEST_TRACK_TIMELINE if x is not None) > 5): klt_score += 10
details['KLT'] = klt_score
raw += klt_score

# --- Deep SORT (30) ---
ds_score = 0
if len(DS_ID_TIMELINE) > 0:
    # минимальная проверка: есть не только None
    if any(x is not None for x in DS_ID_TIMELINE):
        ds_score = 30
details['DeepSORT'] = ds_score
raw += ds_score

# --- Metrics (20) ---
met_score = 0
if KLT_SEGS and KLT_CONT_AVG > 1: met_score += 10
if DS_SEGS and DS_CONT_AVG > 1:   met_score += 10
details['Metrics'] = met_score
raw += met_score

raw_score=raw



In [10]:
import json

# применяем штраф
try:
    pf = penalty_fraction(start_dt, due_dt, submission_dt)
except NameError:
    from datetime import timezone
    pf = 0.0
# ✅ Итоговый результат


final_score = max(0.0, raw_score * (1.0 - min(1.0, pf)))

print(f"Сырой балл: {raw_score}/{100}")
print(f"Штраф (доля): {pf:.4f}")
print(f"Итоговый балл после штрафа: {final_score:.2f}/{100}")

# Последняя строка — JSON, который читает harness
final = {
    "name": full_name,
    "group": student_group,
    "assignment": assignment_id,
    "score": float(final_score)
}

Сырой балл: 40/100
Штраф (доля): 0.0000
Итоговый балл после штрафа: 40.00/100
